# 1-Hour Ahead Prediction Engine (Net Flow Direct V3)

This notebook generates **station-level predictions for the next full hour** using the production-ready XGBoost model selected during training.

It combines:

- the latest hourly station flow state
- recent historical net flow behavior
- target-hour weather
- target-hour event context
- cyclic time features

to estimate:
- next-hour net flow
- expected bikes available
- expected docks available
- station-level operational risk

The main output is:

- `workspace.default.current_station_predictions_next_hour_netflow_direct_v3_cyclic_stationtrend`

This notebook represents the **short-term operational prediction engine** of the forecasting pipeline.

## Process Overview

This notebook performs the following steps:

### 1. Load model artifacts and metadata
The notebook reads:
- feature metadata
- best-model metadata
- the production XGBoost model

This ensures serving uses exactly the same feature contract defined during training.

### 2. Read the current hourly operational table
The notebook loads:
- `station_hour_flow_current`

This table provides the latest observed station state and recent hourly operational history.

### 3. Identify the latest closed hour and define the target hour
The model uses the most recent high-quality closed hour as the reference point and predicts the **next full hour**.

### 4. Rebuild historical features for serving
Recent station history is used to reconstruct:
- lag features
- rolling net flow statistics
- station recent activity features
- cyclic time variables

### 5. Build target-hour context
For the prediction hour, the notebook joins:
- target-hour weather
- target-hour event features
- station-level state at the latest closed hour

### 6. Score the XGBoost model
Using the full serving feature contract, the notebook generates direct next-hour net flow predictions for each downtown station.

### 7. Convert model output into business-facing predictions
The net flow prediction is transformed into:
- predicted bikes available
- predicted docks available
- bike delta
- risk labels
- risk score

### 8. Save the final hourly prediction table
The resulting prediction output is written into a Delta table for operational use.

In [0]:
%pip install xgboost==2.0.3 
%restart_python 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T
from pyspark.sql.types import IntegerType
import pandas as pd
import numpy as np
import json
import zlib
import xgboost as xgb
import datetime as dt

# ============================================================
# UNIFIED JOB 1H - NET FLOW DIRECT V3
#
# Flow:
#   1) Read current hourly flow table
#   2) Build target-hour serving features
#   3) Use target-hour weather + target-hour public events
#   4) Score XGBoost model
#   5) Build business output for NEXT FULL HOUR
#   6) Save final prediction table
# ============================================================

# ------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------
FLOW_TBL = "workspace.default.station_hour_flow_current"

WEATHER_TARGET_VIEW = "workspace.default.vw_weather_hourly_minimal_latest"

EVENTS_DETAIL_DIR = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/"
    "public_events_hourly_detail"
)

FEATURE_META_JSON_DBFS = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata/"
    "netflow_direct_v3_cyclic_stationtrend/features_netflow_direct_v3_cyclic_stationtrend.json"
)

BEST_MODEL_META_JSON_DBFS = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata/"
    "netflow_direct_v3_cyclic_stationtrend/best_model_netflow_direct_v3_cyclic_stationtrend.json"
)

TARGET_TBL = "workspace.default.current_station_predictions_next_hour_netflow_direct_v3_cyclic_stationtrend"

LOOKBACK_HOURS = 220
MAX_MODEL_BYTES = 30_000_000
MAX_META_BYTES = 500_000

USE_EXPLICIT_TARGET = False
TARGET_Y, TARGET_M, TARGET_D, TARGET_H = None, None, None, None

DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

EVENT_RADIUS_KM = 3.0
EVENT_DECAY_KM = 1.5
EVENT_EPS = 0.10

VERY_HIGH_RADIUS_KM = 0.3
HIGH_RADIUS_KM = 0.6
MEDIUM_RADIUS_KM = 1.5
LOW_RADIUS_KM = 3.0

VERY_HIGH_MULT = 1.8
HIGH_MULT = 1.4
MEDIUM_MULT = 1.0
LOW_MULT = 0.6

PI = 3.141592653589793

# ------------------------------------------------------------
# 1) HELPERS
# ------------------------------------------------------------
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def read_dbfs_text(dbfs_path: str, max_bytes: int) -> str:
    return dbutils.fs.head(dbfs_path, max_bytes)

def station_to_bucket(station_id: str, n_buckets: int) -> int:
    if station_id is None:
        return 0
    return zlib.crc32(str(station_id).encode("utf-8")) % n_buckets

def hour_key_from_parts(y: int, m: int, d: int, h: int) -> str:
    return f"{y:04d}{m:02d}{d:02d}{h:02d}"

def risk_label(pred_bikes, pred_docks):
    if pred_bikes <= 1:
        return "CRITICAL_EMPTY"
    elif pred_bikes <= 3:
        return "LOW_BIKES"
    elif pred_docks <= 1:
        return "CRITICAL_FULL"
    elif pred_docks <= 3:
        return "LOW_DOCKS"
    else:
        return "NORMAL"

def load_xgb_booster_from_dbfs_json(dbfs_path: str, max_bytes: int = MAX_MODEL_BYTES) -> xgb.Booster:
    s = read_dbfs_text(dbfs_path, max_bytes=max_bytes)
    if len(s) < 1000:
        raise Exception(f"Model JSON looks too small or truncated: {dbfs_path}")
    booster = xgb.Booster()
    booster.load_model(bytearray(s.encode("utf-8")))
    return booster

def cyc_hour(hour):
    angle = 2.0 * np.pi * hour / 24.0
    return np.sin(angle), np.cos(angle)

def cyc_dow(dow_num):
    dow_idx = dow_num - 1
    angle = 2.0 * np.pi * dow_idx / 7.0
    return np.sin(angle), np.cos(angle)

def cyc_month(month):
    month_idx = month - 1
    angle = 2.0 * np.pi * month_idx / 12.0
    return np.sin(angle), np.cos(angle)

def haversine_matrix_km(st_lat, st_lon, ev_lat, ev_lon):
    st_lat = np.radians(np.asarray(st_lat, dtype=np.float64))[:, None]
    st_lon = np.radians(np.asarray(st_lon, dtype=np.float64))[:, None]
    ev_lat = np.radians(np.asarray(ev_lat, dtype=np.float64))[None, :]
    ev_lon = np.radians(np.asarray(ev_lon, dtype=np.float64))[None, :]

    dlat = ev_lat - st_lat
    dlon = ev_lon - st_lon

    a = np.sin(dlat / 2.0) ** 2 + np.cos(st_lat) * np.cos(ev_lat) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return 6371.0 * c

def tier_multiplier_matrix(dist_km):
    return np.select(
        [
            dist_km <= VERY_HIGH_RADIUS_KM,
            dist_km <= HIGH_RADIUS_KM,
            dist_km <= MEDIUM_RADIUS_KM,
            dist_km <= LOW_RADIUS_KM
        ],
        [
            VERY_HIGH_MULT,
            HIGH_MULT,
            MEDIUM_MULT,
            LOW_MULT
        ],
        default=0.0
    )

def build_event_features_for_target_hour(
    pdf_target_stations: pd.DataFrame,
    pdf_events_detail: pd.DataFrame,
    target_ts: pd.Timestamp
) -> pd.DataFrame:
    """
    Build event features for the target hour, by station.
    """
    out = pdf_target_stations.copy()

    out["event_day_flag"] = 0
    out["events_day_count"] = 0
    out["event_day_attendance_sum"] = 0.0
    out["event_active_nearby_flag"] = 0
    out["events_nearby_count"] = 0
    out["nearest_event_km"] = 999.0
    out["event_weighted_intensity"] = 0.0
    out["event_attendance_est_sum_nearby"] = 0.0
    out["event_impact_score"] = 0.0

    if pdf_events_detail.empty:
        return out

    target_date = pd.Timestamp(target_ts.date())

    pdf_day = (
        pdf_events_detail.loc[
            (pdf_events_detail["start_date"] <= target_date) &
            (pdf_events_detail["end_date"] >= target_date),
            ["event_id", "attendance_est"]
        ]
        .drop_duplicates()
    )

    if not pdf_day.empty:
        out["event_day_flag"] = 1
        out["events_day_count"] = int(pdf_day["event_id"].nunique())
        out["event_day_attendance_sum"] = float(pdf_day["attendance_est"].sum())

    pdf_active = pdf_events_detail.loc[
        pdf_events_detail["target_ts_hour"] == target_ts,
        ["event_lat", "event_lon", "attendance_est"]
    ].copy()

    if pdf_active.empty:
        return out

    st_lat = pd.to_numeric(out["lat"], errors="coerce").fillna(0.0).values
    st_lon = pd.to_numeric(out["lon"], errors="coerce").fillna(0.0).values

    ev_lat = pd.to_numeric(pdf_active["event_lat"], errors="coerce").fillna(0.0).values
    ev_lon = pd.to_numeric(pdf_active["event_lon"], errors="coerce").fillna(0.0).values
    ev_att = pd.to_numeric(pdf_active["attendance_est"], errors="coerce").fillna(0.0).values

    dist_km = haversine_matrix_km(st_lat, st_lon, ev_lat, ev_lon)

    within = dist_km <= EVENT_RADIUS_KM

    nearest_km = dist_km.min(axis=1)
    nearby_count = within.sum(axis=1)
    nearby_flag = (nearby_count > 0).astype(int)

    attendance_nearby = within.astype(np.float64) @ ev_att

    inv_dist_weighted = np.where(within, ev_att[None, :] / (dist_km + EVENT_EPS), 0.0)
    event_weighted_intensity = inv_dist_weighted.sum(axis=1)

    tier_mult = tier_multiplier_matrix(dist_km)
    spillover_component = np.where(
        within,
        ev_att[None, :] * np.exp(-dist_km / EVENT_DECAY_KM) * tier_mult,
        0.0
    )
    spillover_score = spillover_component.sum(axis=1)

    out["event_active_nearby_flag"] = nearby_flag
    out["events_nearby_count"] = nearby_count.astype(int)
    out["nearest_event_km"] = np.where(nearby_flag > 0, nearest_km, 999.0)
    out["event_weighted_intensity"] = event_weighted_intensity.astype(float)
    out["event_attendance_est_sum_nearby"] = attendance_nearby.astype(float)
    out["event_impact_score"] = spillover_score.astype(float)

    return out

# ------------------------------------------------------------
# 2) LOAD METADATA + MODEL
# ------------------------------------------------------------
if not path_exists(FEATURE_META_JSON_DBFS):
    raise Exception(f"Feature metadata not found: {FEATURE_META_JSON_DBFS}")

if not path_exists(BEST_MODEL_META_JSON_DBFS):
    raise Exception(f"Best model metadata not found: {BEST_MODEL_META_JSON_DBFS}")

feat_meta = json.loads(read_dbfs_text(FEATURE_META_JSON_DBFS, MAX_META_BYTES))
best_meta = json.loads(read_dbfs_text(BEST_MODEL_META_JSON_DBFS, MAX_META_BYTES))

FEATURES = feat_meta["netflow_features"]
HASH_BUCKETS = int(feat_meta.get("hash_buckets", 512))
BEST_MODEL = best_meta["best_model"]

if BEST_MODEL != "xgboost":
    raise Exception(f"This unified 1H job currently supports only xgboost. Best model found: {BEST_MODEL}")

MODEL_PATH = best_meta["xgb_model_path"]
booster = load_xgb_booster_from_dbfs_json(MODEL_PATH)

print("Loaded metadata + model")
print("Best model   :", BEST_MODEL)
print("Feature count:", len(FEATURES))
print("Hash buckets :", HASH_BUCKETS)

# ------------------------------------------------------------
# 3) READ CURRENT HOURLY FLOW
# ------------------------------------------------------------
df = spark.table(FLOW_TBL)

required_cols = {
    "station_id", "year", "month", "day", "hour",
    "estimated_departures", "estimated_arrivals",
    "temperature_2m_celsius", "apparent_temperature_celsius",
    "date", "dow_num", "is_weekend",
    "event_day_flag", "events_day_count", "event_day_attendance_sum",
    "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby", "event_impact_score",
    "hour_complete_flag", "flow_quality_flag",
    "lat", "lon",
    "hour_end_snapshot_ts_local",
    "num_bikes_available_end_hour",
    "num_docks_available_end_hour",
    "capacity",
    "name"
}

missing = sorted(list(required_cols - set(df.columns)))
if missing:
    raise Exception(f"{FLOW_TBL} missing required columns: {missing}")

rows_before_bbox = df.count()

df = df.filter(
    (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
    (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
)

rows_after_bbox = df.count()

print("Rows before downtown bounding:", rows_before_bbox)
print("Rows after downtown bounding :", rows_after_bbox)

# ------------------------------------------------------------
# 4) BUILD NET FLOW BASE
# ------------------------------------------------------------
df = (
    df
    .withColumn(
        "net_flow",
        (F.col("estimated_arrivals") - F.col("estimated_departures")).cast("double")
    )
    .withColumn("abs_net_flow", F.abs(F.col("net_flow")))
    .withColumn(
        "ts_hour",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.col("date").cast("string"),
                F.format_string("%02d:00:00", F.col("hour"))
            )
        )
    )
)

df_base = df

df_hist = df.filter(
    (F.col("hour_complete_flag") == 1) &
    (F.col("flow_quality_flag") == 1)
)

# ------------------------------------------------------------
# 5) DETERMINE TARGET HOUR
# ------------------------------------------------------------
if USE_EXPLICIT_TARGET:
    target_ts = spark.sql(
        f"SELECT timestamp('{TARGET_Y:04d}-{TARGET_M:02d}-{TARGET_D:02d} {TARGET_H:02d}:00:00') AS ts"
    ).collect()[0]["ts"]

    latest_closed_ts = spark.sql(
        f"SELECT timestamp('{TARGET_Y:04d}-{TARGET_M:02d}-{TARGET_D:02d} {TARGET_H:02d}:00:00') - INTERVAL 1 HOUR AS ts"
    ).collect()[0]["ts"]
else:
    latest_closed_ts = df_hist.select(F.max("ts_hour").alias("mx")).collect()[0]["mx"]

    if latest_closed_ts is None:
        raise Exception("Could not determine latest closed hour from current hourly flow table.")

    target_ts = spark.sql(
        f"SELECT timestamp('{str(latest_closed_ts)}') + INTERVAL 1 HOUR AS ts"
    ).collect()[0]["ts"]

target_parts = spark.createDataFrame([(target_ts,)], ["ts"]).select(
    F.year("ts").alias("y"),
    F.month("ts").alias("m"),
    F.dayofmonth("ts").alias("d"),
    F.hour("ts").alias("h")
).collect()[0]

hour_key = hour_key_from_parts(
    int(target_parts["y"]),
    int(target_parts["m"]),
    int(target_parts["d"]),
    int(target_parts["h"])
)

print("Latest closed hour:", latest_closed_ts)
print("Target hour       :", target_ts)
print("Hour key          :", hour_key)

# ------------------------------------------------------------
# 6) LIMIT HISTORY WINDOW
# ------------------------------------------------------------
start_ts = spark.sql(
    f"SELECT timestamp('{str(latest_closed_ts)}') - INTERVAL {LOOKBACK_HOURS} HOURS AS st"
).collect()[0]["st"]

df_hist = df_hist.filter(
    (F.col("ts_hour") >= F.lit(start_ts)) &
    (F.col("ts_hour") <= F.lit(latest_closed_ts))
)

# ------------------------------------------------------------
# 7) BUILD HISTORICAL FEATURES
# ------------------------------------------------------------
df_hist = (
    df_hist
    .withColumn("hour_angle", F.lit(2.0 * PI) * F.col("hour") / F.lit(24.0))
    .withColumn("hour_sin", F.sin("hour_angle"))
    .withColumn("hour_cos", F.cos("hour_angle"))
    .withColumn("dow_idx", F.col("dow_num") - F.lit(1))
    .withColumn("dow_angle", F.lit(2.0 * PI) * F.col("dow_idx") / F.lit(7.0))
    .withColumn("dow_sin", F.sin("dow_angle"))
    .withColumn("dow_cos", F.cos("dow_angle"))
    .withColumn("month_idx", F.col("month") - F.lit(1))
    .withColumn("month_angle", F.lit(2.0 * PI) * F.col("month_idx") / F.lit(12.0))
    .withColumn("month_sin", F.sin("month_angle"))
    .withColumn("month_cos", F.cos("month_angle"))
)

w = Window.partitionBy("station_id").orderBy(F.col("ts_hour"))
roll_w_3h = w.rowsBetween(-3, -1)
roll_w_24h = w.rowsBetween(-24, -1)

df_hist = (
    df_hist
    .withColumn("lag1_net", F.lag("net_flow", 1).over(w))
    .withColumn("lag2_net", F.lag("net_flow", 2).over(w))
    .withColumn("lag24_net", F.lag("net_flow", 24).over(w))
    .withColumn("lag168_net", F.lag("net_flow", 168).over(w))
    .withColumn("roll_mean_3h_net", F.avg("net_flow").over(roll_w_3h))
    .withColumn("roll_std_24h_net", F.stddev("net_flow").over(roll_w_24h))
    .withColumn("station_mean_24h_net", F.avg("net_flow").over(roll_w_24h))
    .withColumn("station_abs_mean_24h_net", F.avg("abs_net_flow").over(roll_w_24h))
    .withColumn("station_std_24h_net", F.stddev("net_flow").over(roll_w_24h))
)

# ------------------------------------------------------------
# 8) BUILD TARGET-HOUR BASE ROWS
# ------------------------------------------------------------
df_last_state = df_base.filter(F.col("ts_hour") == F.lit(latest_closed_ts)).select(
    "station_id",
    "name",
    "lat",
    "lon",
    "capacity",
    "hour_end_snapshot_ts_local",
    "num_bikes_available_end_hour",
    "num_docks_available_end_hour"
)

if df_last_state.count() == 0:
    raise Exception("No latest closed station state found.")

target_parts_full = spark.createDataFrame([(target_ts,)], ["ts"]).select(
    F.year("ts").alias("year"),
    F.month("ts").alias("month"),
    F.dayofmonth("ts").alias("day"),
    F.to_date("ts").alias("date"),
    F.hour("ts").alias("hour"),
    F.dayofweek(F.to_date("ts")).alias("dow_num")
).collect()[0]

df_target = (
    df_last_state
    .withColumn("year", F.lit(int(target_parts_full["year"])))
    .withColumn("month", F.lit(int(target_parts_full["month"])))
    .withColumn("day", F.lit(int(target_parts_full["day"])))
    .withColumn("hour", F.lit(int(target_parts_full["hour"])))
    .withColumn("date", F.lit(target_parts_full["date"]))
    .withColumn("dow_num", F.lit(int(target_parts_full["dow_num"])))
    .withColumn("is_weekend", F.when(F.col("dow_num").isin([1, 7]), 1).otherwise(0))
)

# ------------------------------------------------------------
# 8.1) TARGET-HOUR WEATHER
# ------------------------------------------------------------
df_weather_target = spark.table(WEATHER_TARGET_VIEW).select(
    "year",
    "month",
    "day",
    "hour",
    F.col("temperature_2m_c").alias("temperature_2m_celsius"),
    F.col("apparent_temperature_c").alias("apparent_temperature_celsius")
)

df_target = df_target.join(
    df_weather_target,
    on=["year", "month", "day", "hour"],
    how="left"
)

df_target = (
    df_target
    .withColumn("temperature_2m_celsius", F.coalesce(F.col("temperature_2m_celsius"), F.lit(0.0)))
    .withColumn("apparent_temperature_celsius", F.coalesce(F.col("apparent_temperature_celsius"), F.lit(0.0)))
)

# ------------------------------------------------------------
# 8.2) TARGET-HOUR EVENTS
# ------------------------------------------------------------
if not path_exists(EVENTS_DETAIL_DIR):
    raise Exception(f"Events detail input not found: {EVENTS_DETAIL_DIR}")

pdf_events_detail = spark.read.parquet(EVENTS_DETAIL_DIR).select(
    "target_ts_hour",
    "event_id",
    "start_date",
    "end_date",
    "event_lat",
    "event_lon",
    "attendance_est"
).toPandas()

if not pdf_events_detail.empty:
    pdf_events_detail["target_ts_hour"] = pd.to_datetime(pdf_events_detail["target_ts_hour"], errors="coerce")
    pdf_events_detail["start_date"] = pd.to_datetime(pdf_events_detail["start_date"], errors="coerce").dt.normalize()
    pdf_events_detail["end_date"] = pd.to_datetime(pdf_events_detail["end_date"], errors="coerce").dt.normalize()
    pdf_events_detail["attendance_est"] = pd.to_numeric(pdf_events_detail["attendance_est"], errors="coerce").fillna(0.0)
    pdf_events_detail["event_lat"] = pd.to_numeric(pdf_events_detail["event_lat"], errors="coerce")
    pdf_events_detail["event_lon"] = pd.to_numeric(pdf_events_detail["event_lon"], errors="coerce")

pdf_target_stations = df_target.select(
    "station_id", "name", "lat", "lon", "capacity",
    "year", "month", "day", "hour", "date", "dow_num", "is_weekend",
    "temperature_2m_celsius", "apparent_temperature_celsius",
    "hour_end_snapshot_ts_local",
    "num_bikes_available_end_hour", "num_docks_available_end_hour"
).toPandas()

pdf_target_stations = build_event_features_for_target_hour(
    pdf_target_stations=pdf_target_stations,
    pdf_events_detail=pdf_events_detail,
    target_ts=pd.Timestamp(target_ts)
)

pdf_target_stations = pdf_target_stations.replace({np.nan: None})

if "hour_end_snapshot_ts_local" in pdf_target_stations.columns:
    pdf_target_stations["hour_end_snapshot_ts_local"] = pd.to_datetime(
        pdf_target_stations["hour_end_snapshot_ts_local"], errors="coerce"
    )
    pdf_target_stations["hour_end_snapshot_ts_local"] = pdf_target_stations["hour_end_snapshot_ts_local"].apply(
        lambda x: x.strftime("%Y-%m-%d %H:%M:%S") if pd.notna(x) else None
    )

if "date" in pdf_target_stations.columns:
    pdf_target_stations["date"] = pd.to_datetime(pdf_target_stations["date"], errors="coerce")
    pdf_target_stations["date"] = pdf_target_stations["date"].apply(
        lambda x: x.strftime("%Y-%m-%d") if pd.notna(x) else None
    )

target_records = pdf_target_stations.to_dict("records")

target_schema = T.StructType([
    T.StructField("station_id", T.StringType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("lat", T.DoubleType(), True),
    T.StructField("lon", T.DoubleType(), True),
    T.StructField("capacity", T.LongType(), True),
    T.StructField("year", T.LongType(), True),
    T.StructField("month", T.LongType(), True),
    T.StructField("day", T.LongType(), True),
    T.StructField("hour", T.LongType(), True),
    T.StructField("date", T.StringType(), True),
    T.StructField("dow_num", T.LongType(), True),
    T.StructField("is_weekend", T.LongType(), True),
    T.StructField("temperature_2m_celsius", T.DoubleType(), True),
    T.StructField("apparent_temperature_celsius", T.DoubleType(), True),
    T.StructField("hour_end_snapshot_ts_local", T.StringType(), True),
    T.StructField("num_bikes_available_end_hour", T.LongType(), True),
    T.StructField("num_docks_available_end_hour", T.LongType(), True),
    T.StructField("event_day_flag", T.LongType(), True),
    T.StructField("events_day_count", T.LongType(), True),
    T.StructField("event_day_attendance_sum", T.DoubleType(), True),
    T.StructField("event_active_nearby_flag", T.LongType(), True),
    T.StructField("events_nearby_count", T.LongType(), True),
    T.StructField("nearest_event_km", T.DoubleType(), True),
    T.StructField("event_weighted_intensity", T.DoubleType(), True),
    T.StructField("event_attendance_est_sum_nearby", T.DoubleType(), True),
    T.StructField("event_impact_score", T.DoubleType(), True),
])

df_target = spark.createDataFrame(target_records, schema=target_schema)

df_target = (
    df_target
    .withColumn("date", F.to_date("date"))
    .withColumn("hour_end_snapshot_ts_local", F.to_timestamp("hour_end_snapshot_ts_local"))
)

# ------------------------------------------------------------
# 8.3) TARGET-HOUR CYCLIC FEATURES
# ------------------------------------------------------------
df_target = (
    df_target
    .withColumn("hour_angle", F.lit(2.0 * PI) * F.col("hour") / F.lit(24.0))
    .withColumn("hour_sin", F.sin("hour_angle"))
    .withColumn("hour_cos", F.cos("hour_angle"))
    .withColumn("dow_idx", F.col("dow_num") - F.lit(1))
    .withColumn("dow_angle", F.lit(2.0 * PI) * F.col("dow_idx") / F.lit(7.0))
    .withColumn("dow_sin", F.sin("dow_angle"))
    .withColumn("dow_cos", F.cos("dow_angle"))
    .withColumn("month_idx", F.col("month") - F.lit(1))
    .withColumn("month_angle", F.lit(2.0 * PI) * F.col("month_idx") / F.lit(12.0))
    .withColumn("month_sin", F.sin("month_angle"))
    .withColumn("month_cos", F.cos("month_angle"))
)

# ------------------------------------------------------------
# 9) TAKE HISTORICAL FEATURE VALUES FROM LATEST CLOSED HOUR
# ------------------------------------------------------------
df_hist_prev = df_hist.filter(F.col("ts_hour") == F.lit(latest_closed_ts)).select(
    "station_id",
    F.col("net_flow").alias("lag1_net"),
    F.col("lag1_net").alias("lag2_net"),
    F.col("lag24_net").alias("lag24_net"),
    F.col("lag168_net").alias("lag168_net"),
    "roll_mean_3h_net",
    "roll_std_24h_net",
    "station_mean_24h_net",
    "station_abs_mean_24h_net",
    "station_std_24h_net"
)

df_serv = df_target.join(df_hist_prev, on="station_id", how="inner")

# ------------------------------------------------------------
# 10) IMPUTE LONG LAGS / STATS
# ------------------------------------------------------------
df_serv = (
    df_serv
    .withColumn("lag168_net", F.coalesce(F.col("lag168_net"), F.col("lag24_net"), F.col("lag1_net"), F.lit(0.0)))
    .withColumn("lag24_net", F.coalesce(F.col("lag24_net"), F.col("lag1_net"), F.lit(0.0)))
    .withColumn("lag2_net", F.coalesce(F.col("lag2_net"), F.col("lag1_net"), F.lit(0.0)))
    .withColumn("roll_std_24h_net", F.coalesce(F.col("roll_std_24h_net"), F.lit(0.0)))
    .withColumn("station_mean_24h_net", F.coalesce(F.col("station_mean_24h_net"), F.col("lag1_net"), F.lit(0.0)))
    .withColumn("station_abs_mean_24h_net", F.coalesce(F.col("station_abs_mean_24h_net"), F.abs(F.col("lag1_net")), F.lit(0.0)))
    .withColumn("station_std_24h_net", F.coalesce(F.col("station_std_24h_net"), F.lit(0.0)))
)

# ------------------------------------------------------------
# 11) STATION BUCKET
# ------------------------------------------------------------
station_to_bucket_udf = F.udf(
    lambda s: int(station_to_bucket(s, HASH_BUCKETS)) if s is not None else None,
    IntegerType()
)

df_serv = df_serv.withColumn("station_bucket", station_to_bucket_udf(F.col("station_id")))

# ------------------------------------------------------------
# 12) ENSURE FULL FEATURE CONTRACT
# ------------------------------------------------------------
all_required = sorted(list(set(
    FEATURES + [
        "station_id", "name", "lat", "lon", "capacity",
        "year", "month", "day", "hour", "date", "dow_num", "is_weekend",
        "hour_end_snapshot_ts_local",
        "num_bikes_available_end_hour", "num_docks_available_end_hour"
    ]
)))

for c in all_required:
    if c not in df_serv.columns:
        if c in [
            "station_bucket", "month", "hour", "dow_num", "is_weekend",
            "event_day_flag", "events_day_count", "event_active_nearby_flag", "events_nearby_count"
        ]:
            df_serv = df_serv.withColumn(c, F.lit(0))
        else:
            df_serv = df_serv.withColumn(c, F.lit(0.0))

# ------------------------------------------------------------
# 13) DROP ROWS WITHOUT CRITICAL FEATURES
# ------------------------------------------------------------
critical_cols = [
    "temperature_2m_celsius",
    "apparent_temperature_celsius",
    "lag1_net",
    "lag2_net",
    "roll_mean_3h_net",
    "hour_sin",
    "hour_cos",
    "station_mean_24h_net"
]

rows_raw = df_serv.count()
df_serv = df_serv.dropna(subset=critical_cols)
rows_final = df_serv.count()

if rows_final == 0:
    raise Exception(
        "No serving rows left after dropna. There is not enough usable flow history "
        "to compute the required netflow features."
    )

print("Serving rows (raw target):", rows_raw)
print("Serving rows (after dropna):", rows_final)

# ------------------------------------------------------------
# 14) SCORE MODEL
# ------------------------------------------------------------
pdf_score = df_serv.select(*all_required).toPandas()
pdf_score = pdf_score.sort_values(["station_id"]).reset_index(drop=True)

X_df = pdf_score[FEATURES].copy()
X = X_df.astype(np.float32).values

dmat = xgb.DMatrix(X, feature_names=FEATURES)
pred = booster.predict(dmat).astype(np.float32)

pdf_score["net_pred"] = pred
pdf_score["model_version"] = "netflow_direct_v3_cyclic_stationtrend"
pdf_score["best_model"] = BEST_MODEL

# ------------------------------------------------------------
# 15) BUSINESS OUTPUT - NEXT FULL HOUR
# ------------------------------------------------------------
pdf_score["hour_ts_local"] = pd.to_datetime(pdf_score["date"], errors="coerce") + pd.to_timedelta(pdf_score["hour"], unit="h")
pdf_score["predicted_for_hour_local"] = pdf_score["hour_ts_local"] + pd.Timedelta(hours=1)
pdf_score["hour_end_snapshot_ts_local"] = pd.to_datetime(pdf_score["hour_end_snapshot_ts_local"], errors="coerce")

# Keep the column for compatibility, but for this job we predict the FULL next hour
pdf_score["remaining_minutes_in_hour"] = 60.0

# Model output already represents the next FULL hour
pdf_score["net_pred_full_hour"] = pdf_score["net_pred"].astype(float)
pdf_score["net_pred"] = pdf_score["net_pred_full_hour"]

pdf_score["predicted_bikes_next_hour_raw"] = (
    pd.to_numeric(pdf_score["num_bikes_available_end_hour"], errors="coerce").fillna(0.0)
    + pd.to_numeric(pdf_score["net_pred"], errors="coerce").fillna(0.0)
)

cap = pd.to_numeric(pdf_score["capacity"], errors="coerce").fillna(0.0)
pdf_score["predicted_bikes_next_hour"] = np.clip(pdf_score["predicted_bikes_next_hour_raw"], 0.0, cap)
pdf_score["predicted_docks_next_hour"] = cap - pdf_score["predicted_bikes_next_hour"]

pdf_score["delta_bikes_next_hour"] = (
    pdf_score["predicted_bikes_next_hour"]
    - pd.to_numeric(pdf_score["num_bikes_available_end_hour"], errors="coerce").fillna(0.0)
)

pdf_score["risk_level"] = [
    risk_label(b, d)
    for b, d in zip(pdf_score["predicted_bikes_next_hour"], pdf_score["predicted_docks_next_hour"])
]

def risk_score_fn(x):
    if x in ["CRITICAL_EMPTY", "CRITICAL_FULL"]:
        return 4
    elif x in ["LOW_BIKES", "LOW_DOCKS"]:
        return 3
    return 1

pdf_score["risk_score"] = pdf_score["risk_level"].map(risk_score_fn)

for c in [
    "remaining_minutes_in_hour",
    "net_pred_full_hour",
    "net_pred",
    "predicted_bikes_next_hour",
    "predicted_docks_next_hour",
    "delta_bikes_next_hour"
]:
    pdf_score[c] = pd.to_numeric(pdf_score[c], errors="coerce").round(4 if "net_pred" in c else 2)

# ------------------------------------------------------------
# 16) BUILD FINAL SPARK OUTPUT
# ------------------------------------------------------------
for c in ["hour_ts_local", "predicted_for_hour_local", "hour_end_snapshot_ts_local", "date"]:
    if c in pdf_score.columns:
        pdf_score[c] = pd.to_datetime(pdf_score[c], errors="coerce")

pdf_score = pdf_score.replace({np.nan: None})

for c in ["hour_ts_local", "predicted_for_hour_local", "hour_end_snapshot_ts_local"]:
    if c in pdf_score.columns:
        pdf_score[c] = pdf_score[c].apply(
            lambda x: x.to_pydatetime() if pd.notna(x) else None
        )

if "date" in pdf_score.columns:
    pdf_score["date"] = pdf_score["date"].apply(
        lambda x: None if pd.isna(x) else (x.date() if isinstance(x, dt.datetime) else x)
    )

final_cols = [
    "station_id",
    "name",
    "lat",
    "lon",
    "capacity",
    "year",
    "month",
    "day",
    "hour",
    "date",
    "hour_ts_local",
    "hour_end_snapshot_ts_local",
    "predicted_for_hour_local",
    "remaining_minutes_in_hour",
    "num_bikes_available_end_hour",
    "num_docks_available_end_hour",
    "temperature_2m_celsius",
    "apparent_temperature_celsius",
    "event_day_flag",
    "events_day_count",
    "event_day_attendance_sum",
    "event_active_nearby_flag",
    "events_nearby_count",
    "nearest_event_km",
    "event_weighted_intensity",
    "event_attendance_est_sum_nearby",
    "event_impact_score",
    "net_pred_full_hour",
    "net_pred",
    "predicted_bikes_next_hour",
    "predicted_docks_next_hour",
    "delta_bikes_next_hour",
    "risk_level",
    "risk_score",
    "model_version",
    "best_model"
]

final_pdf = pdf_score[final_cols].copy()

def to_safe_timestamp_str(x):
    if pd.isna(x):
        return None
    if isinstance(x, pd.Timestamp):
        return x.strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(x, dt.datetime):
        return x.strftime("%Y-%m-%d %H:%M:%S")
    return str(x)

def to_safe_date_str(x):
    if pd.isna(x):
        return None
    if isinstance(x, pd.Timestamp):
        return x.strftime("%Y-%m-%d")
    if isinstance(x, dt.datetime):
        return x.strftime("%Y-%m-%d")
    if isinstance(x, dt.date):
        return x.strftime("%Y-%m-%d")
    return str(x)

for c in ["hour_ts_local", "hour_end_snapshot_ts_local", "predicted_for_hour_local"]:
    if c in final_pdf.columns:
        final_pdf[c] = final_pdf[c].apply(to_safe_timestamp_str)

if "date" in final_pdf.columns:
    final_pdf["date"] = final_pdf["date"].apply(to_safe_date_str)

final_schema = T.StructType([
    T.StructField("station_id", T.StringType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("lat", T.DoubleType(), True),
    T.StructField("lon", T.DoubleType(), True),
    T.StructField("capacity", T.LongType(), True),
    T.StructField("year", T.LongType(), True),
    T.StructField("month", T.LongType(), True),
    T.StructField("day", T.LongType(), True),
    T.StructField("hour", T.LongType(), True),
    T.StructField("date", T.StringType(), True),
    T.StructField("hour_ts_local", T.StringType(), True),
    T.StructField("hour_end_snapshot_ts_local", T.StringType(), True),
    T.StructField("predicted_for_hour_local", T.StringType(), True),
    T.StructField("remaining_minutes_in_hour", T.DoubleType(), True),
    T.StructField("num_bikes_available_end_hour", T.LongType(), True),
    T.StructField("num_docks_available_end_hour", T.LongType(), True),
    T.StructField("temperature_2m_celsius", T.DoubleType(), True),
    T.StructField("apparent_temperature_celsius", T.DoubleType(), True),
    T.StructField("event_day_flag", T.LongType(), True),
    T.StructField("events_day_count", T.LongType(), True),
    T.StructField("event_day_attendance_sum", T.DoubleType(), True),
    T.StructField("event_active_nearby_flag", T.LongType(), True),
    T.StructField("events_nearby_count", T.LongType(), True),
    T.StructField("nearest_event_km", T.DoubleType(), True),
    T.StructField("event_weighted_intensity", T.DoubleType(), True),
    T.StructField("event_attendance_est_sum_nearby", T.DoubleType(), True),
    T.StructField("event_impact_score", T.DoubleType(), True),
    T.StructField("net_pred_full_hour", T.DoubleType(), True),
    T.StructField("net_pred", T.DoubleType(), True),
    T.StructField("predicted_bikes_next_hour", T.DoubleType(), True),
    T.StructField("predicted_docks_next_hour", T.DoubleType(), True),
    T.StructField("delta_bikes_next_hour", T.DoubleType(), True),
    T.StructField("risk_level", T.StringType(), True),
    T.StructField("risk_score", T.LongType(), True),
    T.StructField("model_version", T.StringType(), True),
    T.StructField("best_model", T.StringType(), True),
])

final_records = final_pdf.to_dict("records")
final_sdf = spark.createDataFrame(final_records, schema=final_schema)

final_sdf = (
    final_sdf
    .withColumn("date", F.to_date("date"))
    .withColumn("hour_ts_local", F.to_timestamp("hour_ts_local"))
    .withColumn("hour_end_snapshot_ts_local", F.to_timestamp("hour_end_snapshot_ts_local"))
    .withColumn("predicted_for_hour_local", F.to_timestamp("predicted_for_hour_local"))
    .withColumn(
        "forecast_generated_from_hour",
        F.to_timestamp(F.lit(str(latest_closed_ts)))
    )
    .withColumn("scored_at_utc", F.current_timestamp())
)

print("=== SCHEMA FINAL CLEANED ===")
final_sdf.printSchema()

(
    final_sdf
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TBL)
)

print("Unified 1h prediction table created:", TARGET_TBL)
print("Rows:", final_sdf.count())

display(
    final_sdf.orderBy("station_id").limit(30)
)

## Outputs

This notebook produces the following table:

- `workspace.default.current_station_predictions_next_hour_netflow_direct_v3_cyclic_stationtrend`

### Output Characteristics
The table contains one row per downtown station for the next target hour and includes:

- station identifiers and metadata
- prediction target timestamp
- latest observed station state
- weather conditions
- event-related contextual features
- predicted full-hour net flow
- predicted bikes available
- predicted docks available
- station-level risk classification

### Main Prediction Fields
Examples of key output columns include:
- `net_pred_full_hour`
- `net_pred`
- `predicted_bikes_next_hour`
- `predicted_docks_next_hour`
- `delta_bikes_next_hour`
- `risk_level`
- `risk_score`
- `forecast_generated_from_hour`
- `predicted_for_hour_local`

This table is intended for short-term operational decision support.

## Key Insights and Summary

### 1. The notebook predicts the next full hour, not partial remaining minutes
The model output represents the **entire next target hour**, making the prediction consistent with operational scheduling and business interpretation.

This is critical because the output is intended to support hourly rebalancing decisions.

---

### 2. Real-time operational context is aligned with training logic
The notebook reconstructs the same feature structure used during model training, including:
- lag-based net flow features
- cyclic time variables
- weather context
- event influence
- station-level recent behavior

This improves consistency between training and serving.

---

### 3. Weather and event signals are incorporated dynamically
The prediction for each target hour is enriched with:
- target-hour temperature conditions
- target-hour apparent temperature
- nearby event activity
- event spillover impact

This helps the model capture real-world demand shifts rather than relying only on historical momentum.

---

### 4. Predictions are translated into operationally meaningful outputs
Instead of only predicting net flow, the notebook also converts predictions into:
- expected bike availability
- expected dock availability
- station risk status

This makes the output directly usable for operations.

---

### 5. Risk labels improve interpretability
The risk classification logic translates numeric forecasts into clear operational categories such as:
- `NORMAL`
- `LOW_BIKES`
- `LOW_DOCKS`
- `CRITICAL_EMPTY`
- `CRITICAL_FULL`

This improves business usability and makes the forecast easier to act on.

---

### 6. Business relevance
This notebook is the short-term prediction engine that supports:
- near-term station imbalance detection
- empty/full station alerts
- short-term rebalancing prioritization
- operational awareness of the next forecast hour

In practical terms, this notebook converts the trained model into a live, actionable hourly decision-support system.